In [1]:
import os
from pathlib import Path
print(os.getcwd())
DATA_ROOT = (Path.cwd().parent / "data").resolve()
DATA_ROOT

/home/famo00001/kincf/stats


PosixPath('/home/famo00001/kincf/data')

In [2]:
from kinodata.data import KinodataDocked

In [3]:
DATA_ROOT = (Path.cwd().parent / "data").resolve()
data = KinodataDocked(root=str(DATA_ROOT))
# data = KinodataDocked()
data

KinodataDocked(119522)

In [4]:
df = data.df.copy()
df.shape

Reading data frame from /home/famo00001/kincf/data/raw/kinodata_docked_v2.sdf.gz...


UndefinedVariableError: name 'activities' is not defined

In [ ]:
ligand_col = 'compound_structures.canonical_smiles'
kinase_col = 'UniprotID'

pairs = df[[ligand_col, kinase_col]].dropna().drop_duplicates()

n_unique_ligands = pairs[ligand_col].nunique()
n_unique_kinases = pairs[kinase_col].nunique()
n_unique_pairs = len(pairs)

summary = pd.DataFrame({
    'metric': [
        'unique molecules',
        'unique kinases',
        'unique molecule-kinase pairs',
    ],
    'value': [n_unique_ligands, n_unique_kinases, n_unique_pairs],
})
summary

From the .sdf directly

In [ ]:

import pandas as pd
from rdkit.Chem import PandasTools

sdf_path = Path(DATA_ROOT / "raw/kinodata_docked_v2.sdf.gz")

df = PandasTools.LoadSDF(
    str(sdf_path),
    smilesName="compound_structures.canonical_smiles",
    molColName="molecule",
    embedProps=True,
    removeHs=True,
)

# Handle old/new column naming
ligand_col = (
    "compound_structures.canonical_smiles"
    if "compound_structures.canonical_smiles" in df.columns
    else "compound_structures_canonical_smiles"
)
activity_type_col = (
    "activities.standard_type"
    if "activities.standard_type" in df.columns
    else "activities_standard_type"
)
kinase_col = "UniprotID"  # expected in KinodataDocked raw SDF

# # Optional pIC50 subset
# if activity_type_col in df.columns:
#     df = df[df[activity_col] == "pIC50"]

pairs = df[[ligand_col, kinase_col]].dropna().drop_duplicates()

molecules_per_kinase = (
    pairs.groupby(kinase_col)[ligand_col].nunique().sort_values(ascending=False)
)
kinases_per_molecule = (
    pairs.groupby(ligand_col)[kinase_col].nunique().sort_values(ascending=False)
)

print("Unique kinases:", pairs[kinase_col].nunique())
print("Unique molecules:", pairs[ligand_col].nunique())
print("\nMolecules per kinase (describe):")
print(molecules_per_kinase.describe())
print("\nKinases per molecule (describe):")
print(kinases_per_molecule.describe())

display(molecules_per_kinase.head(20).rename("n_molecules").reset_index())
display(kinases_per_molecule.head(20).rename("n_kinases").reset_index())


Unique kinases: 248
Unique molecules: 88528

Molecules per kinase (describe):
count     248.000000
mean      482.713710
std       843.336661
min         1.000000
25%        20.750000
50%       141.000000
75%       597.000000
max      6056.000000
Name: compound_structures.canonical_smiles, dtype: float64

Kinases per molecule (describe):
count    88528.000000
mean         1.352261
std          0.966586
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         66.000000
Name: UniprotID, dtype: float64


,UniprotID,n_molecules
0,P35968,6056
1,P00533,5383
2,O60674,4333
3,P15056,3576
4,P43405,2987
5,Q06187,2711
6,P52333,2590
7,P28482,2538
8,O00329,2431
9,P23458,2342


,compound_structures.canonical_smiles,n_kinases
0,CC1(C)C(=O)N([C@H]2CCc3c(O)cccc32)c2nc(Nc3cccc...,66
1,Cn1cc(-c2cnc3c(-c4csc(C(=O)N[C@@H]5CCCC[C@@H]5...,52
2,O=C(NCC(F)(F)F)c1cc(-c2cnn3cc(-c4ccc(OCC[NH+]5...,46
3,Cc1sc(C(=O)N[C@@H]2[C@H]([NH3+])CCCC2(F)F)cc1-...,46
4,Cn1cc(-c2cnc3c(-c4csc(C(=O)NCC(F)(F)F)c4)cnn3c...,42
5,CCCNC(=O)c1ccc(Nc2nc(NCC(F)(F)F)c3cc[nH]c3n2)cc1,39
6,Nc1ncnc2c1c(-c1cccc(O)c1)cn2[C@H]1C[C@@H](C[NH...,29
7,Cc1ccc(NC(=O)c2ccc(C)c(C(F)(F)F)c2)cc1C#Cc1nn(...,27
8,CC(C)(C)c1cc(NC(=O)Nc2ccc(Nc3ncnc4c3OCCCN4)cc2...,27
9,CCn1cc(-c2ccc3c(c2)CCN3C(=O)Cc2cccc(OC(F)(F)F)...,25


In [13]:
save_dir = DATA_ROOT / "stats"
save_dir.mkdir(parents=True, exist_ok=True)
save_dir

PosixPath('/home/famo00001/kincf/data/stats')

In [14]:
rows = [
        ("unique_molecules", pairs[ligand_col].nunique()),
        ("unique_kinases", pairs[kinase_col].nunique()),
        ("unique_molecule_kinase_pairs", len(pairs)),
        ("molecules_per_kinase_min", int(molecules_per_kinase.min())),
        ("molecules_per_kinase_p25", float(molecules_per_kinase.quantile(0.25))),
        ("molecules_per_kinase_median", float(molecules_per_kinase.median())),
        ("molecules_per_kinase_mean", float(molecules_per_kinase.mean())),
        ("molecules_per_kinase_p75", float(molecules_per_kinase.quantile(0.75))),
        ("molecules_per_kinase_max", int(molecules_per_kinase.max())),
        ("kinases_per_molecule_min", int(kinases_per_molecule.min())),
        ("kinases_per_molecule_p25", float(kinases_per_molecule.quantile(0.25))),
        ("kinases_per_molecule_median", float(kinases_per_molecule.median())),
        ("kinases_per_molecule_mean", float(kinases_per_molecule.mean())),
        ("kinases_per_molecule_p75", float(kinases_per_molecule.quantile(0.75))),
        ("kinases_per_molecule_max", int(kinases_per_molecule.max())),
    ]
report_df = pd.DataFrame(rows, columns=["metric", "value"])
report_df.to_csv(save_dir / "summary_metrics.csv", index=False)
report_df.head()

,metric,value
0,unique_molecules,88528.00
1,unique_kinases,248.00
2,unique_molecule_kinase_pairs,119713.00
3,molecules_per_kinase_min,1.00
4,molecules_per_kinase_p25,20.75


In [8]:
df.columns

Index(['docking.posit_probability', 'docking.chemgauss_score',
       'activities.activity_id', 'assays.chembl_id',
       'target_dictionary.chembl_id', 'molecule_dictionary.chembl_id',
       'molecule_dictionary.max_phase', 'activities.standard_type',
       'activities.standard_value', 'activities.standard_units',
       'compound_structures.canonical_smiles',
       'compound_structures.standard_inchi', 'component_sequences.sequence',
       'assays.confidence_score', 'docs.chembl_id', 'docs.year',
       'docs.authors', 'UniprotID', 'similar.klifs_structure_id',
       'similar.fp_similarity', 'docking.predicted_rmsd', 'ID', 'molecule'],
      dtype='object')

In [ ]:
from stats.plot_dataset_stats import (
    plot_coverage_curve,
    plot_histogram,
    plot_pair_degree_hexbin,
    plot_rank_frequency,
    plot_shared_kinase_heatmap,
)


In [ ]:
pairs_for_plot = pairs.rename(columns={ligand_col: "ligand_id", kinase_col: "kinase_id"})

plot_histogram(
    molecules_per_kinase,
    title="Molecules per kinase",
    xlabel="# molecules for kinase",
    save_path=save_dir / "hist_molecules_per_kinase.png",
)

plot_histogram(
    kinases_per_molecule,
    title="Kinases per molecule",
    xlabel="# kinases for molecule",
    save_path=save_dir / "hist_kinases_per_molecule.png",
)

plot_rank_frequency(
    molecules_per_kinase,
    title="Rank-frequency of molecules per kinase",
    ylabel="# molecules",
    save_path=save_dir / "rank_frequency_molecules_per_kinase.png",
)

plot_rank_frequency(
    kinases_per_molecule,
    title="Rank-frequency of kinases per molecule",
    ylabel="# kinases",
    save_path=save_dir / "rank_frequency_kinases_per_molecule.png",
)

plot_coverage_curve(
    molecules_per_kinase,
    title="Coverage by top kinases",
    ylabel="Cumulative fraction of molecule-kinase pairs",
    save_path=save_dir / "coverage_top_kinases.png",
)

plot_coverage_curve(
    kinases_per_molecule,
    title="Coverage by top molecules",
    ylabel="Cumulative fraction of molecule-kinase pairs",
    save_path=save_dir / "coverage_top_molecules.png",
)

shared = plot_shared_kinase_heatmap(
    pairs_for_plot,
    top_k=20,
    save_path=save_dir / "shared_ligands_top20_kinases.png",
)

edge_view = plot_pair_degree_hexbin(
    pairs_for_plot,
    gridsize=45,
    save_path=save_dir / "edge_degree_hexbin.png",
)

display(shared.head())
display(edge_view.head())
print(f"Saved plots to: {save_dir}")
